In [ ]:
import boto3
print(boto3.__version__)

1.42.33


In [ ]:
DATASET_ARN = "arn:aws:quicksight:eu-central-1:730335657350:dataset/498e463a-4e55-4db6-b832-100cb1eb6741"


MAPPINGS = {
    "security_id": "Security ID *",
    "security_name": "Security name *",
    "security_type": "Security type",
}

portfolio_sheet = sheets_esg.build_portfolio_sheet("dataset", MAPPINGS)

analysis_id = "portfolio-analysis-v100"  

resp = api.deploy_analysis(
    aws_account_id=AWS_ACCOUNT_ID,
    analysis_id=analysis_id,
    name="Portfolio Analysis (Code) V100",
    dataset_arn=DATASET_ARN,
    sheets=[portfolio_sheet],
    update=False, # False si nouvelle analysis
    region=REGION,
)

print("DEPLOY OK:", resp.get("Arn", resp))


NameError: name 'sheets_esg' is not defined

In [ ]:

import boto3
import botocore
import uuid

AWS_ACCOUNT_ID = "730335657350"
REGION = "eu-central-1"
DATASET_ARN = "arn:aws:quicksight:eu-central-1:730335657350:dataset/498e463a-4e55-4db6-b832-100cb1eb6741"


qs = boto3.client("quicksight", region_name=REGION)

TEST_TEMPLATE_ID = f"perm-test-{uuid.uuid4()}"

try:
    qs.create_template(
        AwsAccountId=AWS_ACCOUNT_ID,
        TemplateId=TEST_TEMPLATE_ID,
        Name="PERMISSION_TEST_DO_NOT_USE",
        SourceEntity={
            "SourceAnalysis": {
                "Arn": f"arn:aws:quicksight:{REGION}:{AWS_ACCOUNT_ID}:analysis/DOES_NOT_EXIST",
                "DataSetReferences": [
                    {"DataSetArn": DATASET_ARN, "DataSetPlaceholder": "dataset"}
                ],
            }
        }
    )
    print("✅ CreateTemplate AUTORISÉ (permission OK)")
except botocore.exceptions.ClientError as e:
    code = e.response["Error"]["Code"]
    msg = e.response["Error"]["Message"]
    print("❌ AWS a répondu")
    print("ErrorCode:", code)
    print("Message:", msg)


ParamValidationError: Parameter validation failed:
Invalid length for parameter AwsAccountId, value: 3, valid min length: 12

In [ ]:
# ACCES REQUIS

from sheets_esg import build_portfolio_sheet
import boto3

def create_or_update_dashboard_from_analysis(
    aws_account_id: str,
    
    region: str,
    analysis_arn: str,
    dataset_arn: str,
    template_id: str,
    template_name: str,
    dashboard_id: str,
    dashboard_name: str,
):
    qs = boto3.client("quicksight", region_name=region)

    # créer ou mettre à jour le TEMPLATE depuis l'ANALYSIS
    template_arn = f"arn:aws:quicksight:{region}:{aws_account_id}:template/{template_id}"

    try:
        qs.create_template(
            AwsAccountId=aws_account_id,
            TemplateId=template_id,
            Name=template_name,
            SourceEntity={
                "SourceAnalysis": {
                    "Arn": analysis_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )
    except qs.exceptions.ResourceExistsException:
        qs.update_template(
            AwsAccountId=aws_account_id,
            TemplateId=template_id,
            Name=template_name,
            SourceEntity={
                "SourceAnalysis": {
                    "Arn": analysis_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )

    # créer ou mettre à jour le DASHBOARD depuis le TEMPLATE
    try:
        return qs.create_dashboard(
            AwsAccountId=aws_account_id,
            DashboardId=dashboard_id,
            Name=dashboard_name,
            SourceEntity={
                "SourceTemplate": {
                    "Arn": template_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )
    except qs.exceptions.ResourceExistsException:
        return qs.update_dashboard(
            AwsAccountId=aws_account_id,
            DashboardId=dashboard_id,
            Name=dashboard_name,
            SourceEntity={
                "SourceTemplate": {
                    "Arn": template_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )


        

from esg_general.analysis_api import deploy_analysis
from sheets_esg import build_overview_sheet, build_risk_sheet

if __name__ == "__main__":

    AWS_ACCOUNT_ID = "730335657350"
    DATASET_ARN = "arn:aws:quicksight:eu-central-1:730335657350:dataset/498e463a-4e55-4db6-b832-100cb1eb6741"
    DATASET_ID = "498e463a-4e55-4db6-b832-100cb1eb6741"
    REGION = "eu-central-1"  

    MAPPINGS = {
        "security_id": "Security ID *",
        "security_name": "Security name *",
        "security_type": "Security type",
    }

    #overview_sheet = build_overview_sheet(DATASET_ID, MAPPINGS)
    portfolio_sheet = build_portfolio_sheet("dataset", MAPPINGS)
    #risk_sheet = build_risk_sheet(DATASET_ID, MAPPINGS)
    ANALYSIS_ID = "portfolio-analysis-v5"

    calculated_fields = [
        {
            "Name": "one",
            "Expression": "1",
            "DataSetIdentifier": "dataset"
        }
    ]


    deploy_analysis(
        aws_account_id=AWS_ACCOUNT_ID,
        analysis_id="portfolio-analysis-v5",
        name="Portfolio Dashboard (Code) V5",
        dataset_arn=DATASET_ARN,
        sheets=[portfolio_sheet],
        update=True,
        region=REGION,
    )

    REGION = "eu-central-1" 
    ANALYSIS_ARN = f"arn:aws:quicksight:{REGION}:{AWS_ACCOUNT_ID}:analysis/portfolio-analysis-v5"


    resp_dash = create_or_update_dashboard_from_analysis(
        aws_account_id=AWS_ACCOUNT_ID,
        region=REGION,
        analysis_arn=ANALYSIS_ARN,
        dataset_arn=DATASET_ARN,
        template_id="portfolio-template",
        template_name="Portfolio Template (Code)",
        dashboard_id="portfolio-dashboard",
        dashboard_name="Portfolio Dashboard (Code)",
    )
    print("Dashboard créé / mis à jour :", resp_dash.get("Arn", resp_dash))


TokenRetrievalError: Error when retrieving token from sso: Token has expired and refresh failed

In [ ]:
# test pour les permissions
import boto3
import botocore

AWS_ACCOUNT_ID = "730335657350"   # account id
REGION = "eu-central-1"

qs = boto3.client("quicksight", region_name=REGION)

def test(name, fn):
    try:
        fn()
        print(f"✅ {name}")
    except botocore.exceptions.ClientError as e:
        print(f"❌ {name} → {e.response['Error']['Code']}")

test("ListAnalyses", lambda: qs.list_analyses(AwsAccountId=AWS_ACCOUNT_ID))
test("ListTemplates", lambda: qs.list_templates(AwsAccountId=AWS_ACCOUNT_ID))
test("ListDashboards", lambda: qs.list_dashboards(AwsAccountId=AWS_ACCOUNT_ID))


ParamValidationError: Parameter validation failed:
Invalid length for parameter AwsAccountId, value: 3, valid min length: 12